In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [0]:
orders_data = []

# Heavy skew: customer_id = 1 has 1000 rows
for i in range(1000):
    orders_data.append((1, f"order_{i}"))

# Other customers have very few rows
for cust_id in range(2, 11):
    orders_data.append((cust_id, f"order_{cust_id}"))

orders_df = spark.createDataFrame(
    orders_data,
    ["customer_id", "order_id"]
)


In [0]:
orders_df.groupBy("customer_id").count().show()

In [0]:
from pyspark.sql.functions import *

orders_df = orders_df.withColumn("salt", when(col("customer_id") == 1, floor(rand() * 10)).otherwise(0))


salt_df = spark.range(0,10).withColumnRenamed("id","salt")
customers_df_join = customers_df.crossJoin(salt_df)

join  = orders_df.join(customers_df_join, ["customer_id","salt"])
join.show()


In [0]:
customers_data = [(i, f"Customer_{i}") for i in range(1, 11)]

customers_df = spark.createDataFrame(
    customers_data,
    ["customer_id", "customer_name"]
)

customers_df.show()

In [0]:
normal_join_df = orders_df.join(customers_df, "customer_id")
normal_join_df.count()

In [0]:
SALT_RANGE = 10

orders_salted_df = orders_df.withColumn(
    "salt",
    F.when(
        F.col("customer_id") == 1,
        F.floor(F.rand() * SALT_RANGE)
    ).otherwise(0)
)

orders_salted_df.groupBy("customer_id", "salt").count().show()

In [0]:
salt_df = spark.range(0, SALT_RANGE).withColumnRenamed("id", "salt")

customers_salted_df = customers_df.crossJoin(salt_df)

customers_salted_df.show(10)

In [0]:
salted_join_df = (
    orders_salted_df
    .join(
        customers_salted_df,
        ["customer_id", "salt"]
    )
)

salted_join_df.count()

In [0]:
final_df = salted_join_df.drop("salt")
final_df.show(10)

In [0]:
SALT_RANGE = 5
df = spark.range(1,10).withColumnRenamed("id", "new_id")
df2 = df.withColumn("salt_id", F.rand()*SALT_RANGE)
display(df2)